In [0]:
dbutils.widgets.text("batch_id","")
v_batch_id = dbutils.widgets.get("batch_id")

In [0]:
%run ../0-common/env-config

In [0]:
bronze_table = f"{catalog_name}.{bronze_schema}.races"

silver_table = f"{catalog_name}.{silver_schema}.races"

In [0]:
from pyspark.sql import functions as F

In [0]:
races_df = (
    spark.read.table(bronze_table).filter(F.col("batch_id") == v_batch_id)
    )

In [0]:
races_selected_df = races_df.select(
    F.col("season"),
    F.col("round"),
    F.col("raceName").alias("race_name"),
    F.col("date").alias("race_date"),
    F.col("circuitId").alias("circuit_id"),
    F.col("ingestion_timestamp"),
    F.col("source_file"),
    F.col("batch_id")
)

In [0]:
races_valid_df = races_selected_df.filter(
    F.col("season").isNotNull() & F.col("round").isNotNull()
)

In [0]:
races_distinct_df = races_valid_df.dropDuplicates(["season","round"])

In [0]:
races_final_df = (
    races_distinct_df
    .withColumn("race_name", F.initcap("race_name"))
    .withColumn("created_at", F.current_timestamp())
    .withColumn("updated_at", F.current_timestamp())
    )

In [0]:
if not spark.catalog.tableExists(silver_table):
    (
    races_final_df.write
    .format('delta')
    .mode("overwrite")
    .saveAsTable(silver_table)
    )
else:
    from delta.tables import DeltaTable

    delta_table = DeltaTable.forName(spark, silver_table)
    (
        delta_table.alias("t")
        .merge(
            races_final_df.alias("r"),
            "t.season = r.season AND t.round = r.round"
        )
        .whenMatchedUpdate(
            condition="r.batch_id >= t.batch_id",
            set={
                "race_name": "r.race_name",
                "race_date": "r.race_date",
                "circuit_id": "r.circuit_id",
                "ingestion_timestamp": "r.ingestion_timestamp",
                "source_file": "r.source_file",
                "batch_id": "r.batch_id",
                "updated_at": "r.updated_at"
            }
        )
        .whenNotMatchedInsertAll()
        .execute()
    )